# Final Model Training on Engineered Dataset

Models in this notebook:
1. XGBoost Regressor
2. Polynomial Regression
3. Multiple Linear Regression
4. Stacking Regressor

Dataset: `Dataset/train_data_feature_engineered_final.csv`

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import TargetEncoder, StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

In [7]:
df = pd.read_csv('Dataset/train_data_feature_engineered_final.csv')
target = 'RecommendationCount'

cat_cols = [c for c in ['PriceCurrency', 'DRMNotice', 'ExtUserAcctNotice'] if c in df.columns]
num_cols = [c for c in df.columns if c not in cat_cols + [target]]

X = df[cat_cols + num_cols]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42,
)

print('Train shape:', X_train.shape)
print('Test shape :', X_test.shape)
print('Target skew:', round(y.skew(), 2))

Train shape: (9085, 103)
Test shape : (2272, 103)
Target skew: 68.01


In [8]:
# Target-encode categoricals once, then reuse for all models
if cat_cols:
    encoder = TargetEncoder(target_type='continuous')
    X_train_cat = pd.DataFrame(
        encoder.fit_transform(X_train[cat_cols], y_train),
        columns=cat_cols, index=X_train.index,
    )
    X_test_cat = pd.DataFrame(
        encoder.transform(X_test[cat_cols]),
        columns=cat_cols, index=X_test.index,
    )
else:
    X_train_cat = pd.DataFrame(index=X_train.index)
    X_test_cat = pd.DataFrame(index=X_test.index)

X_train_enc = pd.concat([X_train_cat, X_train[num_cols]], axis=1)
X_test_enc = pd.concat([X_test_cat, X_test[num_cols]], axis=1)

y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

print('Encoded train shape:', X_train_enc.shape)

Encoded train shape: (9085, 103)


In [9]:
results = []

def evaluate_model(name, model, Xtr, Xte, ytr_log, yte, yte_log):
    model.fit(Xtr, ytr_log)
    pred_log = model.predict(Xte)
    pred = np.expm1(pred_log)
    rmse = mean_squared_error(yte, pred) ** 0.5
    results.append({
        'Model': name,
        'R2 (log space)': r2_score(yte_log, pred_log),
        'R2 (orig scale)': r2_score(yte, pred),
        'MAE (orig scale)': mean_absolute_error(yte, pred),
        'RMSE (orig scale)': rmse,
    })

In [ ]:
# 1) XGBoost
xgb_model = XGBRegressor(
    n_estimators=1200,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42,
    n_jobs=-1,
)
evaluate_model('XGBoost Regressor', xgb_model, X_train_enc, X_test_enc, y_train_log, y_test, y_test_log)

# 2) Multiple Linear Regression
lin_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('linreg', LinearRegression()),
])
evaluate_model('Multiple Linear Regression', lin_pipeline, X_train_enc, X_test_enc, y_train_log, y_test, y_test_log)

# 3) Polynomial Regression (degree=2)
poly_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('ridge', Ridge(alpha=5.0, random_state=42)),
])
evaluate_model('Polynomial Regression (deg=2)', poly_pipeline, X_train_enc, X_test_enc, y_train_log, y_test, y_test_log)

# 4) Stacking
stack_model = StackingRegressor(
    estimators=[
        ('xgb', XGBRegressor(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.9,
            colsample_bytree=0.9,
            random_state=42,
            n_jobs=-1,
        )),
        ('rf', RandomForestRegressor(
            n_estimators=400,
            max_depth=14,
            random_state=42,
            n_jobs=-1,
        )),
        ('ridge', Ridge(alpha=10.0)),
    ],
    final_estimator=Ridge(alpha=2.0),
    n_jobs=-1,
    passthrough=True,
)
evaluate_model('Stacking Regressor', stack_model, X_train_enc, X_test_enc, y_train_log, y_test, y_test_log)

In [ ]:
results_df = pd.DataFrame(results).sort_values('R2 (orig scale)', ascending=False).reset_index(drop=True)
results_df = results_df.round(4)
display(results_df)

plt.figure(figsize=(8, 4))
sns.barplot(data=results_df, x='R2 (orig scale)', y='Model')
plt.title('Model Comparison by R2 (original scale)')
plt.tight_layout()
plt.show()